# Fine-tuning a CI/CD English description into a yaml script model with Unsloth

This notebook fine-tunes a small code LLM so that, given a plain-English
description of a CI/CD pipeline, it generates the ready YAML script
for GitHub Actions or GitLab CI.

Each line of the JSONL looks like:
```json
{"platform": "github", "sha": "...", "instruction": "<English description>", "output": "<YAML script>", "tags": [...]}
```

## Step 1 - Install Unsloth

In [1]:
%%capture
import os
!pip install -q unsloth

In [2]:
import torch
from unsloth import FastLanguageModel
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise SystemError("No GPU.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Torch: 2.10.0+cu128 | CUDA available: True
GPU: Tesla T4


## Step 2 - Upload the dataset (train.jsonl, val.jsonl)

In [ ]:
from pathlib import Path
TRAIN_PATH = "/kaggle/input/datasets/michaelrmi/ci-fine-tune/train.jsonl" #this is my path on kaggle, if the notebook to be rerun change it.
VAL_PATH   = "/kaggle/input/datasets/michaelrmi/ci-fine-tune/val.jsonl"
print("train exists:", Path(TRAIN_PATH).exists(), "| val exists:", Path(VAL_PATH).exists())

train exists: True | val exists: True


## Step 3 - Configuration

In [4]:
MODEL_NAME = "unsloth/Qwen2.5-Coder-3B-Instruct"

MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True 

SYSTEM_PROMPT = (
    "You are an expert CI/CD engineer. Given a plain-English description of a "
    "pipeline, output ONLY the complete, valid YAML for the requested platform "
    "(GitHub Actions or GitLab CI). Do not add explanations or markdown fences."
)

INSTRUCTION_PART = "<|im_start|>user\n"
RESPONSE_PART    = "<|im_start|>assistant\n"

## Step 4 - Load the model (4-bit) and add LoRA adapters

In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = LOAD_IN_4BIT,
    dtype          = None,
)

==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

In [6]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
)

Unsloth 2026.6.9 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


## Step 5 - Format the data with the chat template

Each example becomes a 3-message conversation (system / user / assistant) and is
rendered with the model's chat template. We then drop any example that would
exceed `MAX_SEQ_LENGTH` tokens so the model never sees a truncated YAML.

In [7]:
from datasets import load_dataset

def to_text(example):
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

raw_train = load_dataset("json", data_files=TRAIN_PATH, split="train")
raw_val   = load_dataset("json", data_files=VAL_PATH,   split="train")

train_ds = raw_train.map(to_text, remove_columns=raw_train.column_names)
val_ds   = raw_val.map(to_text,   remove_columns=raw_val.column_names)

def fits(example):
    return len(tokenizer(example["text"]).input_ids) <= MAX_SEQ_LENGTH

before = len(train_ds)
train_ds = train_ds.filter(fits)
val_ds   = val_ds.filter(fits)
print(f"Train: kept {len(train_ds)}/{before} after length filter | Val: {len(val_ds)}")
print("\n----- sample formatted example -----\n")
print(train_ds[0]["text"][:700])

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/16383 [00:00<?, ? examples/s]

Map:   0%|          | 0/334 [00:00<?, ? examples/s]

Filter:   0%|          | 0/16383 [00:00<?, ? examples/s]

Filter:   0%|          | 0/334 [00:00<?, ? examples/s]

Train: kept 15643/16383 after length filter | Val: 320

----- sample formatted example -----

<|im_start|>system
You are an expert CI/CD engineer. Given a plain-English description of a pipeline, output ONLY the complete, valid YAML for the requested platform (GitHub Actions or GitLab CI). Do not add explanations or markdown fences.<|im_end|>
<|im_start|>user
Create a GitHub Actions workflow named Squashfs-ify the compilers that can be triggered manually with optional input for specifying things to squash. The workflow uses a self-hosted ubuntu environment, cleans the directory, checks out the code, and then runs a Docker container to squash specified targets using a compilerexplorer/infra image.<|im_end|>
<|im_start|>assistant
name: Squashfs-ify the compilers

on:
  workflow_dispatc


## Step 6 - Build the trainer (train only on the YAML answer)

In [8]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
 model = model,
 tokenizer = tokenizer,
 train_dataset = train_ds,
 eval_dataset = val_ds,
 args = SFTConfig(
     dataset_text_field = "text",
     max_seq_length = MAX_SEQ_LENGTH,
     padding_free = False,               # <-- fixes the ValueError
     dataset_num_proc = 2,
     packing = False,
     per_device_train_batch_size = 2,    
     gradient_accumulation_steps = 4,    
     num_train_epochs = 2,
     #max_steps = 60,
     learning_rate = 2e-4,
     logging_steps = 20,
     eval_strategy = "steps",
     eval_steps = 200,
     save_strategy = "steps",
     save_steps = 200,
     save_total_limit = 2,
     load_best_model_at_end = True,         
     metric_for_best_model = "eval_loss",   
     greater_is_better = False,             
     optim = "adamw_8bit",
     weight_decay = 0.01,
     lr_scheduler_type = "linear",
     seed = 3407,
     output_dir = "outputs",
     report_to = "none",
 ),
)

trainer = train_on_responses_only(
 trainer,
 instruction_part = INSTRUCTION_PART,
 response_part = RESPONSE_PART,
)


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/15643 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/320 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/15643 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/15643 [00:00<?, ? examples/s]

Map (num_proc=8):   0%|          | 0/320 [00:00<?, ? examples/s]

Filter (num_proc=8):   0%|          | 0/320 [00:00<?, ? examples/s]

## Step 7 - Train

In [9]:
import os
gpu = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu.name} | {round(gpu.total_memory/1024**3, 2)} GB total")

# checkpoint
ckpt = (os.path.isdir("outputs") and any(d.startswith("checkpoint-") for d in os.listdir("outputs")))
print("Resuming from checkpoint" if ckpt else "Starting fresh")

trainer_stats = trainer.train(resume_from_checkpoint=ckpt)

used = round(torch.cuda.max_memory_reserved() / 1024**3, 2)
mins = round(trainer_stats.metrics["train_runtime"] / 60, 2)
print(f"\nDone in {mins} min | peak VRAM reserved: {used} GB")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


GPU: Tesla T4 | 14.56 GB total
Starting fresh


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 15,643 | Num Epochs = 2 | Total steps = 3,912
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,Validation Loss
200,0.644749,0.604282
400,0.598528,0.565448
600,0.552595,0.540612
800,0.501111,0.515874
1000,0.524851,0.489628
1200,0.458889,0.468393
1400,0.412367,0.451402
1600,0.458005,0.434967
1800,0.398839,0.420224
2000,0.354217,0.409446


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 


Done in 692.37 min | peak VRAM reserved: 9.19 GB


## Step 8 - Try it out

In [10]:
FastLanguageModel.for_inference(model)

def generate_pipeline(description, max_new_tokens=1024):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": description},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    out = model.generate(
        input_ids=inputs, max_new_tokens=max_new_tokens,
        temperature=0.2, top_p=0.9, do_sample=True, use_cache=True,
    )
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)

example_desc = (
    "Create a GitHub Actions workflow named CI that triggers on push and "
    "pull_request to the main branch. It runs on ubuntu-latest, checks out the "
    "code, sets up Python 3.11, installs dependencies from requirements.txt, and "
    "runs pytest."
)
print(generate_pipeline(example_desc))

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=1024) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn

name: CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  build:

    runs-on: ubuntu-latest

    steps:
      - uses: actions/checkout@v4
      - name: Set up Python 3.11
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"
      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install -r requirements.txt
      - name: Test with pytest
        run: |
          pytest


## Step 9 - Save the model

In [ ]:
#LoRA adapters only (small ~100-200 MB)
model.save_pretrained("ci_lora_model")
tokenizer.save_pretrained("ci_lora_model")
print("Saved LoRA adapters -> ./ci_lora_model")

# 3) GGUF for llama.cpp / Ollama (q4_k_m).
model.save_pretrained_gguf("ci_gguf", tokenizer, quantization_method="q4_k_m")


Unsloth: Restored added_tokens_decoder metadata in ci_lora_model/tokenizer_config.json.


Saved LoRA adapters -> ./ci_lora_model
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/764 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in ci_gguf/tokenizer_config.json.


...


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:17<00:00,  8.89s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:45<00:00, 22.79s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/ci_gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b9827-mix-1f1aaa4 (app-b9827-mix-1f1aaa4-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['ci_gguf_gguf/qwen2.5-coder-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed succe

{'save_directory': 'ci_gguf',
 'gguf_directory': 'ci_gguf_gguf',
 'gguf_files': ['ci_gguf_gguf/qwen2.5-coder-3b-instruct.Q4_K_M.gguf'],
 'modelfile_location': 'ci_gguf_gguf/Modelfile',
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}